In [1]:
import os
import json
import re
from typing import List

import pandas as pd
from tqdm import tqdm

from Drain import LogParser


def get_config(dataset: str) -> dict:
    if dataset == "mall":
        return {
            "data_dir": "../mall",
            "logs_padding": ["No related logs"],
            "log_format": "<Timestamp> <Thread> <Type> <ClassName> - <Content>",
            "regex": [
                {
                    "regex_pattern": r"\d{4}-\d{2}-\d{2}\s\d{2}:\d{2}:\d{2}\.\d{3}",
                    "mask_with": "timestamp",
                }
            ],
            "trace_id_patterns": r"traceId:[A-Z,a-z,0-9]*",
            "span_id_patterns": r"span_id:[A-Z,a-z,0-9]*",
        }
    elif dataset == "train-ticket":
        return {
            "data_dir": "../train-ticket",
            "logs_padding": ["No related logs"],
            "log_format": "<Date> <Time>  <Type> 1 --- <Thread> <ClassName>       : <Content>",
            "regex": [
                {
                    "regex_pattern": r"[0-9a-fA-F\-]{36}",
                    "mask_with": "UUID",
                },
                {
                    "regex_pattern": r"\d{4}-\d{2}-\d{2}\s\d{2}:\d{2}:\d{2}\.\d{3}",
                    "mask_with": "timestamp",
                },
            ],
            "trace_id_patterns": r"traceId:[A-Z,a-z,0-9]*",
            "span_id_patterns": r"span_id:[A-Z,a-z,0-9]*",
        }
    else:
        raise ValueError("dataset must be 'mall' or 'train-ticket'")


class LogProcessor:
    """
    作用：
    1. 读取 data_alignment_1.parquet
    2. 展平 logs 列
    3. 清洗字符串（换行、trace_id、span_id）
    4. 用 Drain 做日志结构化
    5. 把结构化结果恢复到原来的 span 粒度
    6. 保存为 data_alignment_2.parquet
    """

    def __init__(self, config: dict):
        self.config = config
        self.data_dir = config["data_dir"]
        self.logs_padding = config["logs_padding"]
        self.log_format = config["log_format"]
        self.regex = config["regex"]
        self.trace_id_patterns = config["trace_id_patterns"]
        self.span_id_patterns = config["span_id_patterns"]

        self.input_path = os.path.join(self.data_dir, "data_alignment_1.parquet")
        self.output_path = os.path.join(self.data_dir, "data_alignment_2.parquet")
        self.structured_csv_path = os.path.join(self.data_dir, "log_structured.csv")

        self.df: pd.DataFrame | None = None
        self.log_list: List[str] | None = None
        self.log_list_length: List[int] | None = None

    def load_alignment_data(self) -> pd.DataFrame:
        self.df = pd.read_parquet(self.input_path)
        return self.df

    def flatten_logs(self) -> tuple[List[str], List[int]]:
        """
        把每行的 logs(list[str]) 展平成一个大列表，
        同时记录每一行原来有多少条日志，便于后面还原。
        """
        if self.df is None:
            self.load_alignment_data()

        # 在横表中，self.df["logs"]是一个List(List(str))展平成List(str)
        self.log_list = [log_line for log_group in self.df["logs"].to_list() for log_line in log_group]
        # 记录self.df["logs"]中每个str_list的长度，如[2, 1, 3]
        self.log_list_length = [len(log_group) for log_group in self.df["logs"].to_list()]
        return self.log_list, self.log_list_length

    @staticmethod
    def replace_strings(str_list: List[str], regex_pattern: str, mask_with: str) -> List[str]:
        new_list = []
        for item in tqdm(str_list, postfix=f"replace -> {mask_with}"):
            new_item = re.sub(regex_pattern, mask_with, item)
            new_list.append(new_item)
        return new_list

    def clean_logs(self, log_list: List[str]) -> List[str]:
        """
        对日志做预清洗：
        1. 去换行
        2. 屏蔽 trace id
        3. 屏蔽 span id
        """
        log_list = self.replace_strings(log_list, r"\n", " ")
        log_list = self.replace_strings(log_list, self.trace_id_patterns, "TraceID")
        log_list = self.replace_strings(log_list, self.span_id_patterns, "SpanID")
        return log_list

    def split_real_and_padding_logs(self, log_list: List[str]) -> tuple[List[int], List[str]]:
        """
        把真实日志和占位日志分开。
        logs_padding[0] 一般是 'No related logs'
        """
        padding_token = self.logs_padding[0]
        real_log_indices = [idx for idx, log in enumerate(log_list) if log != padding_token]
        real_logs = [log for idx, log in enumerate(log_list) if log != padding_token]
        return real_log_indices, real_logs

    def run_drain_parser(self, real_logs: List[str], reload: bool = False) -> None:
        """
        对真实日志跑 Drain。
        输出 log_structured.csv
        """
        if (not os.path.exists(self.structured_csv_path)) or reload:
            st = 0.5
            depth = 4

            parser = LogParser(
                self.log_format,
                indir="./",
                outdir=self.data_dir,
                depth=depth,
                st=st,
                rex=self.regex,
                logName="log",
            )
            parser.parse(real_logs)

    def rebuild_structured_log_list(
        self,
        real_log_indices: List[int],
        total_log_count: int,
    ) -> List[str]:
        """
        把 Drain 处理后的结果还原回原始展平日志序列的位置。
        没有日志的位置保留占位符。
        """
        df_structured = pd.read_csv(self.structured_csv_path)
        structured_log_content = df_structured["Content"].to_list()

        # 先默认填空或占位
        reconstructed = [
            "" if i in real_log_indices else self.logs_padding[0]
            for i in range(total_log_count)
        ]

        # 再把真实日志位置替换成结构化结果
        for i in range(len(real_log_indices)):
            reconstructed[real_log_indices[i]] = structured_log_content[i]

        return reconstructed

    @staticmethod
    def build_split_ranges(lengths: List[int]) -> List[List[int]]:
        """
        根据每个 span 原有日志数量，构造切分区间。
        例如 lengths=[2,3] -> [[0,1],[2,3,4]]
        """
        split_ranges = []
        start = 0
        for length in tqdm(lengths, postfix="build split ranges"):
            end = start + length
            split_ranges.append(list(range(start, end)))
            start = end
        return split_ranges

    @staticmethod
    def restore_grouped_logs(flat_logs: List[str], split_ranges: List[List[int]]) -> List[List[str]]:
        """
        把展平日志恢复成每个 span 一组日志。
        """
        grouped_logs = []
        for indices in split_ranges:
            if len(indices) == 0:
                grouped_logs.append([])
            else:
                grouped_logs.append(flat_logs[indices[0]: indices[-1] + 1])
        return grouped_logs

    def save(self) -> str:
        if self.df is None:
            raise ValueError("No dataframe to save.")
        # 得到一个宽表，每一行是一个span对应的数据,日志解析为模板
        # | TraceID | SpanId | ServiceIp | Timestamp | RpcName | host_cpu_xxx | pod_memory_xxx | ... | logs |
        self.df.to_parquet(self.output_path, index=False)
        print(f"save processed log data to: {self.output_path}")
        return self.output_path

    def run(self, reload: bool = False) -> pd.DataFrame:
        """
        完整流程：
        data_alignment_1.parquet
            ↓
        flatten logs
            ↓
        clean logs
            ↓
        Drain parse
            ↓
        rebuild grouped logs
            ↓
        data_alignment_2.parquet
        """
        # 得到一个宽表，每一行是一个span对应的数据
        # | TraceID | SpanId | ServiceIp | Timestamp | RpcName | host_cpu_xxx | pod_memory_xxx | ... | logs |
        self.load_alignment_data()
        log_list, log_list_length = self.flatten_logs()

        # 去除空格，对traceid和spanid进行屏蔽
        cleaned_logs = self.clean_logs(log_list)
        # 筛选非空白日志占用符的日志列表
        real_log_indices, real_logs = self.split_real_and_padding_logs(cleaned_logs)

        self.run_drain_parser(real_logs, reload=reload)

        structured_flat_logs = self.rebuild_structured_log_list(
            real_log_indices=real_log_indices,
            total_log_count=len(cleaned_logs),
        )

        split_ranges = self.build_split_ranges(log_list_length)
        restored_logs = self.restore_grouped_logs(structured_flat_logs, split_ranges)

        self.df["logs"] = restored_logs
        self.save()

        return self.df



In [3]:
dataset = "train-ticket"   # "mall" 或 "train-ticket"
reload = False

config = get_config(dataset)
processor = LogProcessor(config)
result_df = processor.run(reload=reload)

print("\nprocessed_log_df shape:", result_df.shape)

100%|██████████| 15009/15009 [00:00<00:00, 165137.70it/s, load data]


Processed 100.0% of log lines.Parsing done. [Time taken: 0:00:01.307064]


100%|██████████| 38240/38240 [00:00<00:00, 2914330.61it/s, build split ranges]


save processed log data to: ../train-ticket/data_alignment_2.parquet

processed_log_df shape: (38240, 51)
